# 内置Middleware - HumanInTheLoopMiddleware
- 作用：调用工具签，中断Agent行为，等待用户决策


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from langchain_core.tools import tool

from common import init_simple_dashscope_model

model = init_simple_dashscope_model('qwen-max')


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

## 注册HumanInTheLoopMiddleware - 拦截Agent,询问用户是否授权
- 必须设置checkpointer
- 支持策略: approve, edit, reject

In [15]:
agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    # 必须要有checkpointer记录中断位置，否则无法授权（LangChain会报错）
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "Agent需要您的权限"
                }
            }
        )
    ]
)

checkpointer_config = {
    "configurable": {
        'thread_id': "1"
    }
}

resp = agent.invoke(
    {
        "messages": [
            HumanMessage(content="""
            帮我按步骤执行：
            1. 查询北京的天气
            2. 查询最近的新闻
            3. 给wanggx@gamil.com发送邮件，要校验这个邮箱是否存在，如果不存在，发给wangguoxi@gmail.com
            """)
        ]
    },
    config=checkpointer_config
)

# for msg in resp['messages']:
#     msg.pretty_print()
rprint(resp)

{
    'messages': [
        HumanMessage(
            content='\n            帮我按步骤执行：\n            1. 查询北京的天气\n            2. 查询最近的新闻\n
3. 给wanggx@gamil.com发送邮件，要校验这个邮箱是否存在，如果不存在，发给wangguoxi@gmail.com\n            ',
            additional_kwargs={},
            response_metadata={},
            id='b63df906-f720-4509-a94f-a777a9d30302'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 80,
                    'prompt_tokens': 504,
                    'total_tokens': 584,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-ee29f1d1-b2bc-9d31-8500-07284491c338',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f89a9-6767-7812-86c9-ea2361689077-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_6457b7f2874342489f56c8',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_3e84b8ecd21c4f57b4d284', 'type': 'tool_call'},
                {
                    'name': 'send_email_tool',
                    'args': {
                        'body': '这是一封测试邮件，请查收。',
                        'recipient': 'wanggx@gamil.com',
                        'subject': '测试邮件主题'
                    },
                    'id': 'call_301cbf9167fb44f2b17d5e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 504,
                'output_tokens': 80,
                'total_tokens': 584,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "Tool execution requires approval\n\nTool: get_weather\nArgs: {'city': 
'北京', 'is_forcast': False}"
                    },
                    {
                        'name': 'get_news',
                        'args': {},
                        'description': 'Tool execution requires approval\n\nTool: get_news\nArgs: {}'
                    },
                    {
                        'name': 'send_email_tool',
                        'args': {
                            'body': '这是一封测试邮件，请查收。',
                            'recipient': 'wanggx@gamil.com',
                            'subject': '测试邮件主题'
                        },
                        'description': 'Agent需要您的权限'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='5684269ba61b1415801738ff46fdbed9'
        )
    ]
}

## 模拟人工授权
- 决策数必须和待审批工具数一致
- 授权发生错误，需要在checkpoint回放

In [22]:
from langgraph.types import Command

weather_decision = {
    "type": "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}

news_decision = {
    "type": "approve",
}

send_email_decision = {
    "type" : "approve"
}

# 决策数必须和
decisions = {
    "decisions": []
}

interrupt_list = resp.get('__interrupt__', [])
if interrupt_list:
    interrupt_info = interrupt_list[0]
    action_requests = interrupt_info.value['action_requests']
    for action in action_requests:
        if action['name'] == 'get_weather':
            decisions['decisions'].append(weather_decision)
        elif action['name'] == 'get_news':
            decisions['decisions'].append(news_decision)
        elif action['name'] == 'send_email_tool':
            decisions['decisions'].append(send_email_decision)

## 如果发生错误，需要重发，否则即使修正了变量，也无法执行
# 1. 看历史，找到「还在等 HITL」之前、或 next 指向 model/after_model 相关的状态
history = list(agent.get_state_history(checkpointer_config))
# rprint('history', '-' * 50)
# rprint(history)
# 找仍停在中断前的 checkpoint（常见：values 里已有带 tool_calls 的 AIMessage，且 tasks/next 未走完）
# 实操：选 bad resume 之前的那一帧
before = next(s for s in history if s.interrupts)
# rprint('before', '-' * 50)
# rprint(before)
# 2. 从该 checkpoint 重放 → 会再次 pause，等待新的 resume
resp = agent.invoke(None, config=before.config)

resp = agent.invoke(
    Command(resume=decisions),
    config=checkpointer_config
)

rprint(resp)

{
    'messages': [
        HumanMessage(
            content='\n            帮我按步骤执行：\n            1. 查询北京的天气\n            2. 查询最近的新闻\n
3. 给wanggx@gamil.com发送邮件，要校验这个邮箱是否存在，如果不存在，发给wangguoxi@gmail.com\n            ',
            additional_kwargs={},
            response_metadata={},
            id='b63df906-f720-4509-a94f-a777a9d30302'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 80,
                    'prompt_tokens': 504,
                    'total_tokens': 584,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-ee29f1d1-b2bc-9d31-8500-07284491c338',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f89a9-6767-7812-86c9-ea2361689077-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '上海市', 'is_forcast': True},
                    'id': 'call_6457b7f2874342489f56c8',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_3e84b8ecd21c4f57b4d284', 'type': 'tool_call'},
                {
                    'name': 'send_email_tool',
                    'args': {
                        'body': '这是一封测试邮件，请查收。',
                        'recipient': 'wanggx@gamil.com',
                        'subject': '测试邮件主题'
                    },
                    'id': 'call_301cbf9167fb44f2b17d5e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 504,
                'output_tokens': 80,
                'total_tokens': 584,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='上海市今天天气不错\n明天下雨',
            name='get_weather',
            id='cf8c7629-c4b2-495c-b0ef-26c983eb67c0',
            tool_call_id='call_6457b7f2874342489f56c8'
        ),
        ToolMessage(
            content='中方三艘油轮通过霍尔木兹海峡',
            name='get_news',
            id='ee6e6ea8-d9f9-4e98-8de9-9222a9ee192d',
            tool_call_id='call_3e84b8ecd21c4f57b4d284'
        ),
        ToolMessage(
            content='发送给 wanggx@gamil.com 的邮件标题是：测试邮件主题，内容：这是一封测试邮件，请查收。',
            name='send_email_tool',
            id='299c1310-60bc-4247-90f0-0d90ca4c94c7',
            tool_call_id='call_301cbf9167fb44f2b17d5e'
        ),
        AIMessage(
            content='查询结果如下：\n\n1. 北京市今天的天气情况是不错的，但是请注意明天会有雨。\n\n2. 
最近的新闻头条是：中方三艘油轮通过霍尔木兹海峡。\n\n3. 已经尝试给 wanggx@gamil.com 
发送了一封邮件，邮件标题是“测试邮件主题”，内容为“这是一封测试邮件，请查收。” 
但需要注意的是，邮箱地址可能是错误的（正确的可能是 
wangguoxi@gmail.com），请确认收件人是否收到邮件。如果没有收到，您可能需要使用正确的邮箱地址重发一次。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 650,
                    'total_tokens': 776,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-4ee354d